In [1]:
import pandas as pd

# Load the necessary datasets
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
tags = pd.read_csv('tags.csv')


In [2]:
movies.head(), ratings.head(), tags.head()

(   movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  ,
    userId  movieId  rating     timestamp
 0       1        2     3.5  1.112486e+09
 1       1       29     3.5  1.112485e+09
 2       1       32     3.5  1.112485e+09
 3       1       47     3.5  1.112485e+09
 4       1       50     3.5  1.112485e+09,
    userId  movieId            tag     timestamp
 0      18     4141    Mark Waters  1.240597e+09
 1      65      208      dark hero  1.36

In [3]:
# Display basic info
movies.info(), ratings.info(), tags.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  27278 non-null  int64 
 1   title    27278 non-null  object
 2   genres   27278 non-null  object
dtypes: int64(1), object(2)
memory usage: 639.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500644 entries, 0 to 500643
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     500644 non-null  int64  
 1   movieId    500644 non-null  int64  
 2   rating     500644 non-null  float64
 3   timestamp  500643 non-null  float64
dtypes: float64(2), int64(2)
memory usage: 15.3 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324447 entries, 0 to 324446
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     324447 non-null  int64  
 1   movieId   

(None, None, None)

In [4]:
# Check for missing values
movies.isnull().sum(), ratings.isnull().sum(), tags.isnull().sum()

(movieId    0
 title      0
 genres     0
 dtype: int64,
 userId       0
 movieId      0
 rating       0
 timestamp    1
 dtype: int64,
 userId       0
 movieId      0
 tag          0
 timestamp    1
 dtype: int64)

In [5]:
# Drop rows with missing values
tags.dropna(inplace=True)

In [6]:
# Remove duplicates
movies.drop_duplicates(inplace=True)
ratings.drop_duplicates(inplace=True)
tags.drop_duplicates(inplace=True)

In [7]:
movies.shape, ratings.shape, tags.shape

((27278, 3), (500644, 4), (324446, 4))

In [8]:
# Merge movies and tags data to get movie metadata (i.e., all tags for each movie)
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

# Merge movie_tags with the movies dataset to get the movie title and tags
movie_metadata = pd.merge(movies, movie_tags, on='movieId', how='left')

# View the merged data
movie_metadata.head()


,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Watched computer animation Disney animated fea...
1,2,Jumanji (1995),Adventure|Children|Fantasy,time travel adapted from:book board game child...
2,3,Grumpier Old Men (1995),Comedy|Romance,old people that is actually funny sequel fever...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,chick flick revenge characters chick flick cha...
4,5,Father of the Bride Part II (1995),Comedy,Diane Keaton family sequel Steve Martin weddin...


In [9]:
movie_metadata.shape

(27278, 4)

In [10]:
movie_metadata.isnull().sum()

,0
movieId,0
title,0
genres,0
tag,9571


In [11]:
movie_metadata.dropna(inplace=True)

In [12]:
movie_metadata.shape

(17707, 4)

In [13]:
# --- Popularity-based Filtering ---
# Calculate the number of ratings per movie
movie_ratings_count = ratings.groupby('movieId').size().reset_index(name='num_ratings')

# Calculate the average rating per movie
movie_avg_rating = ratings.groupby('movieId')['rating'].mean().reset_index(name='avg_rating')

# Merge popularity metrics with movies dataset
movies = pd.merge(movies, movie_ratings_count, on='movieId', how='left')
movies = pd.merge(movies, movie_avg_rating, on='movieId', how='left')

# Get top 10 most popular movies based on the number of ratings
popularity_recommendations = movies.sort_values(by='num_ratings', ascending=False).head(10)
print("Top 10 Popular Movies:")
print(popularity_recommendations[['title', 'num_ratings', 'avg_rating']])


Top 10 Popular Movies:
                                          title  num_ratings  avg_rating
293                         Pulp Fiction (1994)       1686.0    4.161625
352                         Forrest Gump (1994)       1670.0    4.048802
587            Silence of the Lambs, The (1991)       1547.0    4.159664
315            Shawshank Redemption, The (1994)       1534.0    4.466428
476                        Jurassic Park (1993)       1483.0    3.664531
257   Star Wars: Episode IV - A New Hope (1977)       1367.0    4.189466
108                           Braveheart (1995)       1334.0    4.058471
583           Terminator 2: Judgment Day (1991)       1269.0    3.938140
2486                         Matrix, The (1999)       1258.0    4.145469
0                              Toy Story (1995)       1241.0    3.975826


In [14]:
!pip install scikit-learn


In [15]:
# --- Content-based Filtering ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Apply TF-IDF Vectorizer on the 'tag' column of the metadata
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movie_metadata['tag'])

# Apply TruncatedSVD for Latent Matrix 1 (Dimensionality reduction)
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix_1 = svd.fit_transform(tfidf_matrix)


In [16]:
# --- Collaborative Filtering ---
# Create a user-movie matrix using a pivot table on ratings
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')

# Replace NaN values with 0 (indicating no rating)
user_movie_matrix = user_movie_matrix.fillna(0)

# Latent Matrix 2 (Collaborative Filtering via SVD)
svd_collab = TruncatedSVD(n_components=50, random_state=42)
latent_matrix_2 = svd_collab.fit_transform(user_movie_matrix)



In [17]:
import numpy as np
# --- Hybrid Filtering ---
# Step 1: Normalize both latent matrices (avoid division by zero)
latent_matrix_1_norm = latent_matrix_1 / (np.linalg.norm(latent_matrix_1, axis=1, keepdims=True) + 1e-10)
latent_matrix_2_norm = latent_matrix_2 / (np.linalg.norm(latent_matrix_2, axis=1, keepdims=True) + 1e-10)

# Step 2: Get the movieId order used in both matrices
movies_in_latent_1 = movie_metadata['movieId'].reset_index(drop=True)
movies_in_latent_2 = user_movie_matrix.columns

# Step 3: Get common movies
common_movies = np.intersect1d(movies_in_latent_1, movies_in_latent_2)

# Step 4: Create movie ID to index mappings
movie_to_idx_1 = {movie_id: idx for idx, movie_id in enumerate(movies_in_latent_1)}
movie_to_idx_2 = {mid: user_movie_matrix.columns.get_loc(mid) for mid in movies_in_latent_2}

# Step 5: Get the indices of common movies in both latent matrices
idx_latent_1 = [movie_to_idx_1[mid] for mid in common_movies if mid in movie_to_idx_1]
idx_latent_2 = [movie_to_idx_2[mid] for mid in common_movies if mid in movie_to_idx_2 and movie_to_idx_2[mid] < latent_matrix_2.shape[0]]

# Step 6: Ensure both lists have the same number of movies
min_length = min(len(idx_latent_1), len(idx_latent_2))

idx_latent_1, idx_latent_2 = idx_latent_1[:min_length], idx_latent_2[:min_length]

# Step 7: Slice both latent matrices using matching indices
latent_matrix_1_common = latent_matrix_1[idx_latent_1, :]
latent_matrix_2_common = latent_matrix_2[idx_latent_2, :]

# Debug: Check shapes
print(f"latent_matrix_1_common shape: {latent_matrix_1_common.shape}")
print(f"latent_matrix_2_common shape: {latent_matrix_2_common.shape}")

# Ensure matrices have the same shape before averaging
assert latent_matrix_1_common.shape == latent_matrix_2_common.shape, "Shape mismatch!"

# Step 8: Average the two matrices to get a hybrid representation
hybrid_matrix = (latent_matrix_1_common + latent_matrix_2_common) / 2

# Step 9: Compute similarity between movies using dot product
similarity_matrix = np.dot(hybrid_matrix, hybrid_matrix.T)

# Step 10: Build the recommender function
def get_hybrid_recommendations(movie_id, top_n=10):
    try:
        # Get the index of the movie in the hybrid matrix
        idx = movies_in_latent_1[movies_in_latent_1 == movie_id].index[0]
        sim_scores = list(enumerate(similarity_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
        movie_indices = [i[0] for i in sim_scores]
        return movie_metadata['title'].iloc[movie_indices]
    except IndexError:
        return f"Movie ID {movie_id} not found in hybrid model."

# Example: Get recommendations for movieId = 1
get_hybrid_recommendations(1)


latent_matrix_1_common shape: (2863, 50)
latent_matrix_2_common shape: (2863, 50)


,title
1008,Robin Hood: Prince of Thieves (1991)
1835,Madeline (1998)
1557,"Hunt for Red October, The (1990)"
753,I Shot Andy Warhol (1996)
1922,"Mask of Zorro, The (1998)"
998,Swiss Family Robinson (1960)
929,To Be or Not to Be (1942)
787,Lone Star (1996)
57,"Postman, The (Postino, Il) (1994)"
158,Congo (1995)


In [18]:
get_hybrid_recommendations(100)

,title
2753,West Beirut (West Beyrouth) (1998)
998,Swiss Family Robinson (1960)
1495,Ponette (1996)
2637,Mystery Men (1999)
1235,"Deer Hunter, The (1978)"
996,Homeward Bound: The Incredible Journey (1993)
1008,Robin Hood: Prince of Thieves (1991)
2448,Escape from the Planet of the Apes (1971)
751,Heavy (1995)
2738,On the Ropes (1999)
